In [1]:
import os

In [2]:
import random

In [3]:
import numpy as np

In [4]:
import pandas as pd

In [5]:
import matplotlib.pyplot as plt

In [6]:
import seaborn as sns

In [7]:
import warnings

In [36]:
from sklearn.model_selection import KFold

In [8]:
from sklearn.linear_model import LogisticRegression

In [9]:
from sklearn.tree import DecisionTreeClassifier

In [10]:
from sklearn.ensemble import RandomForestClassifier

In [11]:
from sklearn.preprocessing import StandardScaler

In [38]:
from sklearn.pipeline import make_pipeline

In [12]:
from xgboost import XGBClassifier

In [43]:
from sklearn.metrics import roc_auc_score, log_loss

In [13]:
warnings.filterwarnings('ignore')

In [14]:
# -----------------------------------------------------------------------------
# CONFIGURACIÓN GLOBAL
# -----------------------------------------------------------------------------
SEED = 42
PRESUPUESTO_MAX = 2000000.0  # Límite presupuestario estricto de $2,000,000 USD
def fijar_semillas(seed=SEED):
    """Garantiza la reproducibilidad exacta fijando semillas globales."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

fijar_semillas(SEED)
# Configuración visual para EDA
sns.set_theme(style='whitegrid', palette='colorblind', font='DejaVu Sans')
CHART_DPI = 150  # Resolución para guardar los gráficos


In [15]:
def cargar_datos(path_train='solicitudes_train.csv', 
                 path_test='solicitudes_test.csv', 
                 path_bancario='comportamiento_bancario.csv'):
    """
    Carga los tres archivos CSV de la competencia.
    """
    print("--- 1. Carga de Datos ---")
    train = pd.read_csv(path_train) if os.path.exists(path_train) else None
    test = pd.read_csv(path_test) if os.path.exists(path_test) else None
    bancario = pd.read_csv(path_bancario) if os.path.exists(path_bancario) else None
    
    if train is not None:
        print(f"train shape: {train.shape}")
    if test is not None:
        print(f"test shape: {test.shape}")
    if bancario is not None:
        print(f"bancario shape: {bancario.shape}")
        
    return train, test, bancario

In [16]:
df_train,df_test,df_bancario=cargar_datos()

--- 1. Carga de Datos ---
train shape: (32416, 8)
test shape: (3061, 7)
bancario shape: (661139, 4)


In [17]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32416 entries, 0 to 32415
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID_Solicitud      32416 non-null  int64  
 1   ID_Usuario        32416 non-null  int64  
 2   Monto_Solicitado  32416 non-null  int64  
 3   Tasa_Interes      32416 non-null  float64
 4   Ingreso_Mensual   32416 non-null  int64  
 5   Motivo_Prestamo   32416 non-null  object 
 6   Fecha_Solicitud   32416 non-null  object 
 7   Estado_Final      32416 non-null  int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 2.0+ MB


In [18]:
df_bancario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 661139 entries, 0 to 661138
Data columns (total 4 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID_Usuario                661139 non-null  int64  
 1   Mes_Referencia            661139 non-null  object 
 2   Saldo_Promedio_Mensual    661139 non-null  float64
 3   Dias_Atraso_Otras_Deudas  661139 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 20.2+ MB


In [19]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3061 entries, 0 to 3060
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID_Solicitud      3061 non-null   int64  
 1   ID_Usuario        3061 non-null   int64  
 2   Monto_Solicitado  3061 non-null   int64  
 3   Tasa_Interes      3061 non-null   float64
 4   Ingreso_Mensual   3061 non-null   int64  
 5   Motivo_Prestamo   3061 non-null   object 
 6   Fecha_Solicitud   3061 non-null   object 
dtypes: float64(1), int64(4), object(2)
memory usage: 167.5+ KB


In [20]:
# EXPLORACION DE DATOS:

In [21]:
def eda_solicitudes(train, output_dir='graficos_eda'):
    """
    Análisis Exploratorio para solicitudes_train.csv
    """
    os.makedirs(output_dir, exist_ok=True)
    print("\n--- 2. Análisis Exploratorio de Solicitudes (Train) ---")
    
    # 2.1 Información General y Nulos
    print("\n> Resumen de Valores Faltantes:")
    nulos = train.isnull().sum()
    if nulos.sum() > 0:
        print(nulos[nulos > 0])
    else:
        print("  ✓ No se encontraron valores faltantes en el conjunto de entrenamiento.")
    
    # 2.2 Balance de Variable Objetivo
    if 'Estado_Final' in train.columns:
        prop = train['Estado_Final'].value_counts(normalize=True) * 100
        print(f"\n> Balance de Clase ('Estado_Final'):")
        print(f"  - Pagados  (1): {prop.get(1, 0):.2f}%")
        print(f"  - Defaults (0): {prop.get(0, 0):.2f}%")
        
        # Gráfico 1: Target Balance
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.countplot(data=train, x='Estado_Final', ax=ax)
        ax.set_title('Distribución de Estado_Final (0: Default, 1: Pagado)', fontsize=12, fontweight='bold')
        
        # Corrección explícita de ticks y labels
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Default (0)', 'Pagado (1)'])
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '01_variable_objetivo.png'), dpi=CHART_DPI)
        plt.close()

    # 2.3 Distribución de Variables Financieras Clave
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Distribución de Variables Financieras Clave en Solicitudes', fontsize=14, fontweight='bold')
    
    if 'Monto_Solicitado' in train.columns:
        sns.histplot(data=train, x='Monto_Solicitado', hue='Estado_Final', kde=True, ax=axes[0], element='step')
        axes[0].set_title('Monto Solicitado')
        
    if 'Tasa_Interes' in train.columns:
        sns.histplot(data=train, x='Tasa_Interes', hue='Estado_Final', kde=True, ax=axes[1], element='step')
        axes[1].set_title('Tasa de Interés')
        
    if 'Ingreso_Mensual' in train.columns:
        sns.histplot(data=train, x='Ingreso_Mensual', hue='Estado_Final', kde=True, ax=axes[2], element='step')
        axes[2].set_title('Ingreso Mensual')
        
    sns.despine()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '02_financieras_dist.png'), dpi=CHART_DPI)
    plt.close()
    print(f"  ✓ Gráficos guardados exitosamente en la carpeta '{output_dir}/'.")

In [22]:
eda_solicitudes(df_train)


--- 2. Análisis Exploratorio de Solicitudes (Train) ---

> Resumen de Valores Faltantes:
  ✓ No se encontraron valores faltantes en el conjunto de entrenamiento.

> Balance de Clase ('Estado_Final'):
  - Pagados  (1): 84.05%
  - Defaults (0): 15.95%
  ✓ Gráficos guardados exitosamente en la carpeta 'graficos_eda/'.


In [23]:
def eda_comportamiento_bancario(bancario, output_dir='graficos_eda'):
    """
    Análisis Exploratorio para comportamiento_bancario.csv
    """
    os.makedirs(output_dir, exist_ok=True)
    print("\n--- Análisis Exploratorio de Comportamiento Bancario ---")
    
    # 3.1 Registros por Usuario
    meses_por_user = bancario.groupby('ID_Usuario')['Mes_Referencia'].count()
    print(f"\n> Registros temporales por usuario:")
    print(f"  - Mínimo de meses: {meses_por_user.min()}")
    print(f"  - Promedio de meses: {meses_por_user.mean():.1f}")
    print(f"  - Máximo de meses: {meses_por_user.max()}")
    
    # 3.2 Días de Atraso en Deudas
    if 'Dias_Atraso_Otras_Deudas' in bancario.columns:
        morosos = (bancario['Dias_Atraso_Otras_Deudas'] > 0).sum()
        pct_morosos = (morosos / len(bancario)) * 100
        print(f"\n> Registros mensuales con atraso (> 0 días): {morosos} de {len(bancario)} ({pct_morosos:.2f}%)")
        
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(bancario['Dias_Atraso_Otras_Deudas'], bins=30, kde=True, ax=ax)
        ax.set_title('Distribución de Días de Atraso en Otras Deudas', fontsize=12, fontweight='bold')
        ax.set_xlabel('Días de Atraso')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '03_atrasos_dist.png'), dpi=CHART_DPI)
        plt.close()
        
    # 3.3 Evaluación del Saldo Promedio
    if 'Saldo_Promedio_Mensual' in bancario.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(x=bancario['Saldo_Promedio_Mensual'], ax=ax)
        ax.set_title('Detección de Outliers en Saldo Promedio Mensual', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '04_saldo_promedio.png'), dpi=CHART_DPI)
        plt.close()
    
    print(f"  ✓ Gráficos bancarios guardados exitosamente en la carpeta '{output_dir}/'.")

In [24]:
eda_comportamiento_bancario(df_bancario)


--- Análisis Exploratorio de Comportamiento Bancario ---

> Registros temporales por usuario:
  - Mínimo de meses: 17
  - Promedio de meses: 22.8
  - Máximo de meses: 24

> Registros mensuales con atraso (> 0 días): 522614 de 661139 (79.05%)
  ✓ Gráficos bancarios guardados exitosamente en la carpeta 'graficos_eda/'.


In [25]:
# INGENIERIA Y CARACTERÍSTICAS:

In [26]:
def agregar_comportamiento_bancario(df_bancario):
    """Consolida hasta 24 meses de historial bancario por usuario."""
    df_bancario = df_bancario.sort_values(by=['ID_Usuario', 'Mes_Referencia'])
    
    # Agregaciones históricas globales (24 meses)
    agg_dict = {
        'Saldo_Promedio_Mensual': ['mean', 'std', 'min', 'max'],
        'Dias_Atraso_Otras_Deudas': ['max', 'mean', lambda x: (x > 0).sum()]
    }
    features_agg = df_bancario.groupby('ID_Usuario').agg(agg_dict)
    features_agg.columns = [
        'saldo_mean', 'saldo_std', 'saldo_min', 'saldo_max',
        'atraso_max', 'atraso_mean', 'meses_con_atraso'
    ]
    
    # Ventanas recientes (3 y 6 meses)
    bancario_3m = df_bancario.groupby('ID_Usuario').tail(3)
    agg_3m = bancario_3m.groupby('ID_Usuario').agg({
        'Saldo_Promedio_Mensual': 'mean',
        'Dias_Atraso_Otras_Deudas': 'max'
    }).rename(columns={'Saldo_Promedio_Mensual': 'saldo_mean_3m', 'Dias_Atraso_Otras_Deudas': 'atraso_max_3m'})
    
    bancario_6m = df_bancario.groupby('ID_Usuario').tail(6)
    agg_6m = bancario_6m.groupby('ID_Usuario').agg({
        'Saldo_Promedio_Mensual': 'mean',
        'Dias_Atraso_Otras_Deudas': 'max'
    }).rename(columns={'Saldo_Promedio_Mensual': 'saldo_mean_6m', 'Dias_Atraso_Otras_Deudas': 'atraso_max_6m'})
    
    features_bancarias = features_agg.join(agg_3m).join(agg_6m).reset_index()
    
    # Ratios de tendencia y estabilidad
    features_bancarias['ratio_saldo_3m_vs_hist'] = features_bancarias['saldo_mean_3m'] / (features_bancarias['saldo_mean'] + 1e-5)
    features_bancarias['deterioro_mora_reciente'] = features_bancarias['atraso_max_3m'] - features_bancarias['atraso_mean']
    features_bancarias['volatilidad_saldo'] = features_bancarias['saldo_std'] / (features_bancarias['saldo_mean'] + 1e-5)
    
    return features_bancarias

In [27]:
agregar_comportamiento_bancario(df_bancario)

,ID_Usuario,saldo_mean,saldo_std,saldo_min,saldo_max,atraso_max,atraso_mean,meses_con_atraso,saldo_mean_3m,atraso_max_3m,saldo_mean_6m,atraso_max_6m,ratio_saldo_3m_vs_hist,deterioro_mora_reciente,volatilidad_saldo
0,1,20017.683913,4291.367162,12768.11,26660.17,59,8.304348,13,24550.633333,57,24299.358333,57,1.226447,48.695652,0.214379
1,2,3781.878571,684.673923,2368.66,4955.03,67,16.619048,21,4515.880000,11,4597.363333,67,1.194084,-5.619048,0.181041
2,3,4577.649545,597.367938,3601.76,5547.47,4,1.090909,12,4556.326667,4,4890.723333,4,0.995342,2.909091,0.130497
3,4,4992.794583,649.225998,3654.98,6122.52,18,1.958333,15,5571.863333,5,5283.425000,5,1.115981,3.041667,0.130033
4,5,2556.866667,642.454050,1753.87,3759.64,70,27.000000,24,1875.490000,19,1948.161667,62,0.733511,-8.000000,0.251266
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28988,28989,2018.809565,454.940657,1241.24,2875.78,36,6.130435,22,1626.690000,6,1580.418333,7,0.805767,-0.130435,0.225351
28989,28990,8671.805500,1254.928224,7216.81,12000.63,16,2.950000,15,8388.570000,6,8582.140000,6,0.967338,3.050000,0.144714
28990,28991,7308.412609,1138.502642,5600.96,9582.84,41,8.565217,22,8683.120000,7,8534.771667,7,1.188099,-1.565217,0.155780
28991,28992,3009.658421,808.379502,1738.30,4774.96,62,10.578947,19,3887.166667,13,3773.386667,13,1.291564,2.421053,0.268595


In [28]:
def construir_features_solicitudes(df_solicitudes, df_bancarias_agg):
    """Cruza solicitudes con agregados bancarios y construye ratios financieros."""
    df = df_solicitudes.merge(df_bancarias_agg, on='ID_Usuario', how='left')
    
    cols_bancarias = [c for c in df_bancarias_agg.columns if c != 'ID_Usuario']
    df[cols_bancarias] = df[cols_bancarias].fillna(0)
    
    # Ratios Financieros de Capacidad de Pago
    df['ratio_monto_ingreso'] = df['Monto_Solicitado'] / (df['Ingreso_Mensual'] + 1e-5)
    df['cuota_estimada'] = (df['Monto_Solicitado'] * (1 + df['Tasa_Interes'])) / 24
    df['ratio_cuota_ingreso'] = df['cuota_estimada'] / (df['Ingreso_Mensual'] + 1e-5)
    df['capacidad_pago_neta'] = df['Ingreso_Mensual'] - df['cuota_estimada']
    
    # Procesamiento de fecha
    if 'Fecha_Solicitud' in df.columns:
        df['Fecha_Solicitud'] = pd.to_datetime(df['Fecha_Solicitud'], errors='coerce')
        df['fecha_mes'] = df['Fecha_Solicitud'].dt.month
        df['fecha_dia_semana'] = df['Fecha_Solicitud'].dt.dayofweek
        df['fecha_es_fin_semana'] = df['fecha_dia_semana'].isin([5, 6]).astype(int)
    
    # Codificación de categóricas
    for col in df.select_dtypes(include=['object', 'category']).columns:
        if col not in ['ID_Solicitud', 'ID_Usuario']:
            df[col] = df[col].astype('category').cat.codes
            
    return df

In [29]:
df_bancario_agg=agregar_comportamiento_bancario(df_bancario) #HistorialBancario_por_usuario

In [30]:
train_full = construir_features_solicitudes(df_train, df_bancario_agg)
test_full = construir_features_solicitudes(df_test, df_bancario_agg)

In [32]:
print('✓ Feature engineering completado:')
print(' - Tamaño Train:', train_full.shape)
print(' - Tamaño Test :', test_full.shape)

✓ Feature engineering completado:
 - Tamaño Train: (32416, 29)
 - Tamaño Test : (3061, 28)


In [33]:
# EVALUACIÓN COMPARATIVA DE MODELOS BASELINE:

In [45]:
def evaluar_modelos_baseline(df_train, features_cols, target_col='Estado_Final', n_splits=5):
    """
    Compara modelos de distinta complejidad (Regresión Logística, Árbol, Random Forest, XGBoost)
    usando KFold de 5 pliegues para establecer una línea base sólida.
    """
    print("\n" + "="*70)
    print(f" EVALUACIÓN COMPARATIVA DE MODELOS (5-Fold Cross-Validation)")
    print("="*70)
    
    X = df_train[features_cols]
    y = df_train[target_col]
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    
    modelos = {
        '1. Regresión Logística (Baseline)': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
        '2. Árbol de Decisión': DecisionTreeClassifier(max_depth=5, random_state=SEED),
        '3. Random Forest': RandomForestClassifier(n_estimators=200, max_depth=6, random_state=SEED, n_jobs=-1),
        '4. XGBoost (Ensamble Avanzado)': XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=SEED, eval_metric='logloss', use_label_encoder=False)
    }
    
    resultados = []
    
    for nombre, model in modelos.items():
        oof_preds = np.zeros(len(df_train))
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
            X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]
            
            model.fit(X_tr, y_tr)
            oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
            
        auc = roc_auc_score(y, oof_preds)
        loss = log_loss(y, oof_preds)
        
        resultados.append({
            'Modelo': nombre,
            'ROC-AUC (OOF)': round(auc, 4),
            'LogLoss (OOF)': round(loss, 4)
        })
        
    df_res = pd.DataFrame(resultados).sort_values(by='ROC-AUC (OOF)', ascending=False)
    print("\nTabla Comparativa de Modelos (Out-of-Fold Metrics):")
    print("-" * 65)
    print(df_res.to_string(index=False))
    print("-" * 65)
    
    return df_res

In [ ]:
# PRUEBA

In [46]:
# 1. Selección de características (excluyendo identificadores y la variable objetivo)
cols_excluir = ['ID_Solicitud', 'ID_Usuario', 'Estado_Final', 'Fecha_Solicitud']
features = [c for c in train_full.columns if c not in cols_excluir]

X = train_full[features]
y = train_full['Estado_Final']

# Configuración del esquema de validación cruzada (5 Folds)
SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

# 2. Definición de modelos para la comparativa (de menor a mayor complejidad)
modelos = {
    '1. Regresión Logística (Baseline)': make_pipeline(
        StandardScaler(), 
        LogisticRegression(max_iter=1000, random_state=SEED)
    ),
    '2. Árbol de Decisión': DecisionTreeClassifier(
        max_depth=5, 
        random_state=SEED
    ),
    '3. Random Forest': RandomForestClassifier(
        n_estimators=200, 
        max_depth=6, 
        random_state=SEED, 
        n_jobs=-1
    ),
    '4. XGBoost (Ensamble Avanzado)': XGBClassifier(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        eval_metric='logloss',
        use_label_encoder=False
    )
}

resultados = []

In [47]:
df_res = evaluar_modelos_baseline (train_full , features)


 EVALUACIÓN COMPARATIVA DE MODELOS (5-Fold Cross-Validation)

Tabla Comparativa de Modelos (Out-of-Fold Metrics):
-----------------------------------------------------------------
                           Modelo  ROC-AUC (OOF)  LogLoss (OOF)
1. Regresión Logística (Baseline)         0.8483         0.3191
   4. XGBoost (Ensamble Avanzado)         0.8465         0.3203
                 3. Random Forest         0.8439         0.3240
             2. Árbol de Decisión         0.8349         0.3364
-----------------------------------------------------------------
